a DataInstanceLibrary should be an index of pointers to DataInstances

this means:
- 2 libs can overlap, or 1 can implicitly contain a subset of the other
- a "bring all here" function can be used to stage (nextflow jobs, from globus, etc.)
- to function as an xgdb, an external "edge" library should describe how data instances are connected

# This has been made obsolete by dev09

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from metasmith.models.libraries import DataInstanceLibrary,DataTypeLibrary, TransformInstanceLibrary
from metasmith.models.solver import Endpoint 
from metasmith.hashing import KeyGenerator

from local.constants import WORKSPACE_ROOT
from local.utils import LinkifyPath
CACHE = WORKSPACE_ROOT/"main/local_mock/cache"

In [ ]:
assert False, "stop notebook"

In [ ]:
from metasmith.coms.ipc import LiveShell

with LiveShell() as shell:
    res = shell.Exec("globus endpoint local-id", history=True)
    out, err = res.out, res.err
    assert len(err) == 0, "\n".join(err)
    assert len(out) == 1, "\n".join(out)
    LOCAL_ID = out[0].strip()
    
LOCAL_ID

In [ ]:
from metasmith.models.remote import GlobusSource

for i, address in enumerate([
    "https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fglobus_test1%2F",
    # "https://g-743d49.88cee.8443.data.globus.org/Metasmith/globus_test1/x.txt", # this requires a search
    "globus://28ea4f3c-8d9d-11ee-8c73-fd88ce9321ad:/home/tony/workspace/tools/Metasmith/main/local_mock/dev5.ipynb"
]):
    x = GlobusSource.Parse(address)
    print(x)
    print(x.endpoint)
    if i == 0: 
        REMOTE_ID = x.endpoint
REMOTE_ID

In [ ]:
batch_path = CACHE/"globus_batch.txt"
REMOTE_PATH = "/Metasmith/globus_test4"
with open(batch_path, "w") as f:
    p = WORKSPACE_ROOT/"scratch/test_ws/data/local"
    lines = []
    for x in p.glob("*"):
        x = Path(x)
        line = f"{x} {REMOTE_PATH}/{x.name}"
        lines.append(line)
    # lines.append(f"{p}/doesnt_exist {REMOTE_PATH}/doesnt_exist\n")
    for line in lines:
        print(line)
        f.write(line+"\n")

In [ ]:
# with LiveShell() as shell:
#     shell.RegisterOnErr(lambda x: print(f"E |{x}"))
#     # res = shell.Exec(f"globus task show -F json {TASK_ID}", history=True)
#     res = shell.Exec(f"globus ls {REMOTE_ID}:{REMOTE_PATH}", history=True)

In [ ]:
res.out

In [ ]:
# src_ep = LOCAL_ID
# dest_ep = REMOTE_ID
# label = "test.dev6"

# TASK_ID = None
# with LiveShell() as shell:
#     def on_out(x):
#         print(f"  |{x}")
#         K = "Task ID"
#         if K in x:
#             global TASK_ID
#             TASK_ID = x.split(" ")[-1].strip()
#     shell.RegisterOnOut(on_out)
#     shell.RegisterOnErr(lambda x: print(f"E |{x}"))
#     cmd = f"globus transfer {src_ep} {dest_ep} --batch {batch_path} --sync-level checksum" + (f" --label {label}" if label else "")
#     res = shell.Exec(cmd, history=True)
# TASK_ID

In [ ]:
# import json

# with LiveShell() as shell:
#     shell.RegisterOnErr(lambda x: print(f"E |{x}"))
#     # res = shell.Exec(f"globus task show -F json {TASK_ID}", history=True)
#     res = shell.Exec(f"globus task event-list --filter-errors -F json {TASK_ID}", history=True)
#     d = json.loads("\n".join(res.out))
# len(d)

In [ ]:
# context_str = d["DATA"][0]["details"]
# context = json.loads(context_str)
# context

In [ ]:
from metasmith.models.remote import GlobusSource

for address in [
    "https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fglobus_test1%2F",
    # "https://g-743d49.88cee.8443.data.globus.org/Metasmith/globus_test1/x.txt", # this requires a search
    "globus://28ea4f3c-8d9d-11ee-8c73-fd88ce9321ad:/home/tony/workspace/tools/Metasmith/main/local_mock/dev5.ipynb"
]:
    x = GlobusSource.Parse(address)
    print(x)

In [ ]:
from metasmith.models.remote import Source, SourceType, Logistics

_kg = KeyGenerator()
label = f"msm.dev6-{_kg.GenerateUID(l=3)}"
import shutil
TEST_DIR = WORKSPACE_ROOT/"main/local_mock/cache/logistics"
# if TEST_DIR.exists(): shutil.rmtree(TEST_DIR)
# TEST_DIR.mkdir(parents=True, exist_ok=True)
mover = Logistics()
# mover.QueueTransfer(
#     src=Source(address="https://g-743d49.88cee.8443.data.globus.org/Metasmith/globus_test1/x.txt", type=SourceType.GLOBUS),
#     dest=Source(address=TEST_DIR/"x.txt", type=SourceType.DIRECT)
# )
# mover.QueueTransfer(
#     src=Source(address="https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fglobus_test1%2F", type=SourceType.GLOBUS),
#     dest=Source(address=TEST_DIR/"test/", type=SourceType.DIRECT)
# )
mover.QueueTransfer(
    src=Source(address=WORKSPACE_ROOT/"scratch/test_ws/data/local/", type=SourceType.DIRECT),
    dest=Source(address="https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fglobus_test5%2F", type=SourceType.GLOBUS),
)
# mover.QueueTransfer(
#     src=Source(address=WORKSPACE_ROOT/"scratch/test_ws/data/local/example.fna", type=SourceType.DIRECT),
#     dest=Source(address=TEST_DIR/"example.symlink.fna", type=SourceType.SYMLINK)
# )
# mover.QueueTransfer(
#     src=Source(address=WORKSPACE_ROOT/"scratch/test_ws/data/local/example.fna", type=SourceType.DIRECT),
#     dest=Source(address=TEST_DIR/"example.direct.fna", type=SourceType.DIRECT)
# )
print(label)
res = mover.ExecuteTransfers(label=label)

In [ ]:
res

In [ ]:
KeyGenerator.FromHex("90c42f49e4b1b50087a48a636716ffa65507af803b19c7973c42d02ced8ac017", l=32)

In [ ]:
KeyGenerator.FromHex("90c42f49e4b1b50087a48a636716ffa65507af803b19c7973c42d02ced8ac017", l=4)